In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

spark = (
    SparkSession.builder
    .appName("binance-etl")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")     # crypto trades in UTC
    .config("spark.driver.memory", "4g")             # avoid OOM on the write
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

In [2]:
schema = StructType([
    StructField("open_time",       LongType(),   True),
    StructField("open",            StringType(), True),
    StructField("high",            StringType(), True),
    StructField("low",             StringType(), True),
    StructField("close",           StringType(), True),
    StructField("volume",          StringType(), True),
    StructField("close_time",      LongType(),   True),
    StructField("quote_volume",    StringType(), True),
    StructField("num_trades",      LongType(),   True),
    StructField("taker_buy_base",  StringType(), True),
    StructField("taker_buy_quote", StringType(), True),
    StructField("ignore",          StringType(), True),
])

In [3]:
files_path = "../data/klines/*/*.csv"

# read all csvs into one df
df = spark.read.csv(files_path, schema=schema, header=False)

# extract Symbol from directories
symbol = F.regexp_extract(F.input_file_name(), r"/klines/([A-Z0-9]+)/", 1)

# create new column Symbol and add symbol to it
df = df.withColumn("symbol", symbol)


print("Total rows:", df.count())

df.sample(fraction=0.0001).show()

Total rows: 11852660
+-------------+--------------+--------------+--------------+--------------+------------+-------------+----------------+----------+--------------+----------------+------+-------+
|    open_time|          open|          high|           low|         close|      volume|   close_time|    quote_volume|num_trades|taker_buy_base| taker_buy_quote|ignore| symbol|
+-------------+--------------+--------------+--------------+--------------+------------+-------------+----------------+----------+--------------+----------------+------+-------+
|1777778880000|78134.96000000|78135.76000000|78134.96000000|78135.76000000|  1.57468000|1777778939999| 123037.94626610|       287|    1.20395000|  94070.96971280|     0|BTCUSDT|
|1778092860000|81354.06000000|81359.39000000|81334.00000000|81334.00000000|  4.46875000|1778092919999| 363546.85535460|      1066|    1.75168000| 142502.95253210|     0|BTCUSDT|
|1778465880000|81287.20000000|81295.47000000|81272.90000000|81276.76000000| 20.14232000|1

In [4]:
# microseconds -> milliseconds -> real timestamp
df = df.withColumn(
    "open_time_ms",
    F.when(F.col("open_time") > 100_000_000_000_000, F.col("open_time") / 1000)
    .otherwise(F.col("open_time"))
    .cast("long")
)

df = df.withColumn(
    "event_time",
    (F.col("open_time_ms") / 1000)
    .cast("timestamp")
)

# numeric casts for the columns we compute on
df = df.withColumn("close_num", F.col("close").cast("double"))
df = df.withColumn("high_num", F.col("high").cast("double"))
df = df.withColumn("low_num", F.col("low").cast("double"))
df = df.withColumn("volume_num", F.col("volume").cast("double"))
df = df.withColumn("taker_buy_base_num", F.col("taker_buy_base").cast("double"))

df.sample(fraction=0.0001).show()

+-------------+--------------+--------------+--------------+--------------+---------------+-------------+----------------+----------+---------------+---------------+------+--------+-------------+-------------------+---------+--------+--------+----------+------------------+
|    open_time|          open|          high|           low|         close|         volume|   close_time|    quote_volume|num_trades| taker_buy_base|taker_buy_quote|ignore|  symbol| open_time_ms|         event_time|close_num|high_num| low_num|volume_num|taker_buy_base_num|
+-------------+--------------+--------------+--------------+--------------+---------------+-------------+----------------+----------+---------------+---------------+------+--------+-------------+-------------------+---------+--------+--------+----------+------------------+
|1778682720000|79810.33000000|79815.99000000|79770.00000000|79774.73000000|    14.78103000|1778682779999|1179285.74010320|      3251|     4.26460000|340252.16920160|     0| BTCUS

#Log-returns#

A log-return is how much the price moved in one minute: `ln(close_now / close_before)`.

We use logs because they add up over time.

 the next feature `Realised variance` relies on that.


In [5]:
by_time = Window.partitionBy("symbol").orderBy("event_time")

df = df.withColumn("prev_close_num", F.lag("close_num").over(by_time))

df = df.withColumn("log_return", F.log(
    F.try_divide(
        F.col("close_num"),
        F.col("prev_close_num")))
)

df = df.withColumn("sq_return", F.col("log_return") * F.col("log_return"))

In [6]:
df.sample(fraction=0.0001).show()

+----------------+----------+----------+----------+----------+----------------+----------------+---------------+----------+---------------+---------------+------+-------+-------------+-------------------+---------+--------+-------+----------+------------------+--------------+--------------------+--------------------+
|       open_time|      open|      high|       low|     close|          volume|      close_time|   quote_volume|num_trades| taker_buy_base|taker_buy_quote|ignore| symbol| open_time_ms|         event_time|close_num|high_num|low_num|volume_num|taker_buy_base_num|prev_close_num|          log_return|           sq_return|
+----------------+----------+----------+----------+----------+----------------+----------------+---------------+----------+---------------+---------------+------+-------+-------------+-------------------+---------+--------+-------+----------+------------------+--------------+--------------------+--------------------+
|1746732900000000|0.25460000|0.25470000|0.2

## Realised variance over 4 windows

`Realised variance` = sum of `squared log-returns` over a time window.

It measures "how much did the price jostle around," ignoring direction. A calm hour = small value; a news spike = large value.

`rowsBetween(-(n-1), 0)` means "this row plus the n-1 rows before it".

In [7]:
def rolling_sum(col, minutes):
  window = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-(minutes - 1), 0)
  return F.sum(col).over(window)

df = df.withColumn("rv_15m",  rolling_sum("sq_return", 15))
df = df.withColumn("rv_1h",   rolling_sum("sq_return", 60))
df = df.withColumn("rv_4h",   rolling_sum("sq_return", 240))
df = df.withColumn("rv_24h",  rolling_sum("sq_return", 1440))

In [8]:
df.show()

+----------------+----------+----------+----------+----------+----------------+----------------+---------------+----------+---------------+---------------+------+-------+-------------+-------------------+---------+--------+-------+----------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|       open_time|      open|      high|       low|     close|          volume|      close_time|   quote_volume|num_trades| taker_buy_base|taker_buy_quote|ignore| symbol| open_time_ms|         event_time|close_num|high_num|low_num|volume_num|taker_buy_base_num|prev_close_num|          log_return|           sq_return|              rv_15m|               rv_1h|               rv_4h|              rv_24h|
+----------------+----------+----------+----------+----------+----------------+----------------+---------------+----------+---------------+---------------+------+-------+-------------+----------

##Parkinson estimator## + ##volume and buy-ratio features##

`Parkinson` uses each candle's high-low range to catch volatility that close-to-close returns miss (a bar that swung wildly but closed flat).

`Volume z-score` = how unusual this minute's volume is vs the last hour.

 `Buy ratio` = how much of the volume was aggressive buying.

 Both feed the anomaly detector later.

In [9]:
def rolling_avg(col, minutes):
  window = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-(minutes - 1), 0)
  return F.avg(col).over(window)

# Parkinson: per-minute, then averaged over a window
factor = 1.0 / (4.0 * F.log( F.lit(2.0) ) )
df = df.withColumn("parkinson_1m", factor * F.pow(F.log(F.col("high_num") / F.col("low_num")), 2))

df = df.withColumn("parkinson_15m", rolling_avg("parkinson_1m", 15))
df = df.withColumn("parkinson_1h",  rolling_avg("parkinson_1m", 60))

# volume z-score over the trailing hour
vol_win = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-59, 0)
df = df.withColumn("vol_mean_1h", F.avg("volume_num").over(vol_win))
df = df.withColumn("vol_std_1h",  F.stddev("volume_num").over(vol_win))
df = df.withColumn("volume_zscore", F.try_divide(
    (F.col("volume_num") - F.col("vol_mean_1h")), F.col("vol_std_1h"))
)

# buy ratio = taker buy volume / total volume
df = df.withColumn("buy_ratio", F.try_divide(
    F.col("taker_buy_base_num"), F.col("volume_num"))
)

In [10]:
df.groupBy("symbol").count().orderBy("symbol").show(25, truncate=False)

total = df.count()
distinct = df.select("symbol", "event_time").distinct().count()
print(f"duplicates: {total - distinct}")

# gaps: more than 60s between consecutive rows
gap_win = Window.partitionBy("symbol").orderBy("event_time")
gaps = (
    df.withColumn("prev_t", F.lag("event_time").over(gap_win))
      .withColumn("gap_s", F.col("event_time").cast("long") - F.col("prev_t").cast("long"))
      .filter(F.col("gap_s") > 60)
)
print("rows with a gap before them:", gaps.count(), "(0 = continuous)")

+--------+------+
|symbol  |count |
+--------+------+
|ADAUSDT |592633|
|APTUSDT |592633|
|ATOMUSDT|592633|
|AVAXUSDT|592633|
|BNBUSDT |592633|
|BTCUSDT |592633|
|DOGEUSDT|592633|
|DOTUSDT |592633|
|ETCUSDT |592633|
|ETHUSDT |592633|
|FILUSDT |592633|
|LINKUSDT|592633|
|LTCUSDT |592633|
|NEARUSDT|592633|
|POLUSDT |592633|
|SOLUSDT |592633|
|TRXUSDT |592633|
|UNIUSDT |592633|
|XLMUSDT |592633|
|XRPUSDT |592633|
+--------+------+

duplicates: 0
rows with a gap before them: 0 (0 = continuous)


In [11]:
def fwd_rv(minutes):
    fwd = (Window.partitionBy("symbol").orderBy("event_time").rowsBetween(1, minutes))
    rv = F.sum("sq_return").over(fwd)
    count = F.count("sq_return").over(fwd)

    return rv, count

rv15, cnt15 = fwd_rv(15)
rv60, cnt60 = fwd_rv(60)

df = (df
    .withColumn("rv_future_15m", rv15)
    .withColumn("cnt_future_15m",  cnt15)
    .withColumn("rv_future_1h",  rv60)
    .withColumn("cnt_future_1h",   cnt60)
)

df = df.filter((F.col("cnt_future_15m") == 15) & (F.col("cnt_future_1h") == 60))

df = (df
    .withColumn("target_15m", F.log1p(F.col("rv_future_15m")))
    .withColumn("target_1h",  F.log1p(F.col("rv_future_1h")))
    .drop("rv_future_15m", "cnt_future_15m", "rv_future_1h", "cnt_future_1h")
)

In [12]:
(
    df.write
    .mode("overwrite")
    .partitionBy("symbol")
    .parquet("../data/klines_processed/features_with_targets")
)
print("saved to ../data/klines_processed/features_with_targets")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_com

Py4JError: An error occurred while calling o306.parquet